# 49. The Causal & Regression-Based Forecasting Problem

## Tier 2: The Classic Heuristic (Stepwise Regression Builder)

### Key assumptions
- Local optimization through greedy feature selection can find good models
- Statistical significance (F-test) effectively identifies important predictors
- Stepwise procedure converges to a parsimonious, interpretable model
- Computational efficiency is more important than global optimality

### Approach (step-by-step)
1. **Initialize** with empty model (forward selection) or all variables (backward elimination)
2. **Evaluate candidates** using F-statistic or information criteria (AIC/BIC)
3. **Select best variable** that provides maximum improvement in model fit
4. **Update model** and repeat until no significant improvements possible
5. **Validate final model** with diagnostic checks and performance metrics

### What to look for in the results
- Step-by-step variable selection process with statistical significance
- Model improvement trajectory (R², AIC, BIC) at each step
- Final parsimonious model with optimal feature subset
- Computational efficiency vs exhaustive search comparison

### Concrete example (from the source)
Synthetic port throughput dataset with multiple potential predictors:
- **Causal variables**: Trade volume, GDP growth, fuel price, seasonal peak, ship density
- **Noise variables**: Random factors that should be excluded
- **Goal**: Automatically identify the truly predictive variables using stepwise regression

In [ ]:
# Import required libraries for stepwise regression
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
# Generate synthetic port throughput data with multiple predictors
np.random.seed(42)  # For reproducible results

n_samples = 100

# True causal variables (should be selected by stepwise regression)
trade_volume = np.random.normal(100, 15, n_samples)
gdp_growth = np.random.normal(2.5, 0.8, n_samples)
fuel_price = np.random.normal(85, 10, n_samples)
seasonal_peak = np.random.binomial(1, 0.3, n_samples)  # 30% peak season
ship_density = np.random.normal(25, 5, n_samples)

# Noise variables (should be excluded by stepwise regression)
random_noise_1 = np.random.normal(0, 1, n_samples)
random_noise_2 = np.random.normal(0, 1, n_samples)
competition = np.random.normal(50, 10, n_samples)  # Not actually causal

# Generate throughput with causal relationships plus noise
# True model: Throughput = 2000 + 15*trade_volume + 300*gdp_growth + 8*fuel_price + 
#             250*seasonal_peak + 20*ship_density + random_error
true_throughput = (
    2000 + 
    15 * trade_volume + 
    300 * gdp_growth + 
    8 * fuel_price + 
    250 * seasonal_peak + 
    20 * ship_density + 
    np.random.normal(0, 200, n_samples)  # Random error
)

# Create DataFrame
data = pd.DataFrame({
    'Trade_Volume': trade_volume,
    'GDP_Growth': gdp_growth,
    'Fuel_Price': fuel_price,
    'Seasonal_Peak': seasonal_peak,
    'Ship_Density': ship_density,
    'Random_Noise_1': random_noise_1,
    'Random_Noise_2': random_noise_2,
    'Competition': competition,
    'Throughput': true_throughput
})

print("Synthetic Port Throughput Dataset:")
print(f"Shape: {data.shape}")
print("\nFirst 5 rows:")
print(data.head().round(2))

print("\nDataset Statistics:")
print(data.describe().round(2))

In [ ]:
def calculate_f_statistic(X_full, X_reduced, y):
    """
    Calculate F-statistic for comparing nested models.
    
    Args:
        X_full: Design matrix for full model (with intercept)
        X_reduced: Design matrix for reduced model (with intercept)
        y: Response vector
    
    Returns:
        f_stat: F-statistic value
        p_value: P-value for F-test
    """
    # Fit both models
    model_full = LinearRegression().fit(X_full, y)
    model_reduced = LinearRegression().fit(X_reduced, y)
    
    # Calculate predictions and residuals
    y_pred_full = model_full.predict(X_full)
    y_pred_reduced = model_reduced.predict(X_reduced)
    
    # Calculate sum of squared errors
    sse_full = np.sum((y - y_pred_full) ** 2)
    sse_reduced = np.sum((y - y_pred_reduced) ** 2)
    
    # Calculate degrees of freedom
    n = len(y)
    df_full = n - X_full.shape[1]
    df_reduced = n - X_reduced.shape[1]
    df_num = df_reduced - df_full  # Numerator degrees of freedom
    
    # Calculate F-statistic
    if df_num > 0 and sse_full > 0:
        f_stat = ((sse_reduced - sse_full) / df_num) / (sse_full / df_full)
        p_value = 1 - stats.f.cdf(f_stat, df_num, df_full)
    else:
        f_stat = 0
        p_value = 1
    
    return f_stat, p_value

def fit_ols_model(X, y):
    """
    Fit OLS model and return coefficients and performance metrics.
    
    Args:
        X: Predictor matrix (with intercept column)
        y: Response vector
    
    Returns:
        Dictionary with model results
    """
    model = LinearRegression().fit(X, y)
    y_pred = model.predict(X)
    
    # Calculate metrics
    mse = mean_squared_error(y, y_pred)
    r2 = r2_score(y, y_pred)
    
    # Calculate AIC and BIC
    n = len(y)
    k = X.shape[1]
    aic = n * np.log(mse) + 2 * k
    bic = n * np.log(mse) + k * np.log(n)
    
    return {
        'coefficients': model.coef_,
        'intercept': model.intercept_,
        'mse': mse,
        'r2': r2,
        'aic': aic,
        'bic': bic,
        'predictions': y_pred
    }

In [ ]:
def forward_stepwise_regression(X, y, significance_level=0.05):
    """
    Implement forward stepwise regression algorithm.
    
    Args:
        X: Predictor matrix (without intercept)
        y: Response vector
        significance_level: Threshold for variable inclusion
    
    Returns:
        Dictionary with stepwise regression results
    """
    n_samples, n_predictors = X.shape
    
    # Initialize
    selected_features = []
    remaining_features = list(range(n_predictors))
    step_results = []
    current_X = np.ones((n_samples, 1))  # Start with intercept only
    
    print("Forward Stepwise Regression Progress:")
    print("=" * 60)
    
    step = 0
    while remaining_features and step < n_predictors:
        best_f_stat = 0
        best_feature = None
        best_p_value = 1
        
        # Test each remaining feature
        for feature in remaining_features:
            # Create candidate model with this feature added
            candidate_X = np.column_stack([current_X, X[:, feature]])
            
            # Calculate F-statistic for adding this feature
            f_stat, p_value = calculate_f_statistic(candidate_X, current_X, y)
            
            # Keep the best feature
            if f_stat > best_f_stat:
                best_f_stat = f_stat
                best_feature = feature
                best_p_value = p_value
        
        # Check if best feature meets significance level
        if best_p_value < significance_level:
            # Add the feature to the model
            selected_features.append(best_feature)
            remaining_features.remove(best_feature)
            current_X = np.column_stack([current_X, X[:, best_feature]])
            
            # Fit model and calculate metrics
            model_results = fit_ols_model(current_X, y)
            
            # Store step results
            step_result = {
                'step': step + 1,
                'feature_added': best_feature,
                'feature_name': data.columns[best_feature],
                'f_statistic': best_f_stat,
                'p_value': best_p_value,
                'selected_features': selected_features.copy(),
                'mse': model_results['mse'],
                'r2': model_results['r2'],
                'aic': model_results['aic'],
                'bic': model_results['bic']
            }
            step_results.append(step_result)
            
            print(f"Step {step + 1}: Added {data.columns[best_feature]}")
            print(f"  F-statistic: {best_f_stat:.4f}")
            print(f"  p-value: {best_p_value:.6f}")
            print(f"  R-squared: {model_results['r2']:.4f}")
            print()
            
            step += 1
        else:
            # No more features meet significance level
            print(f"No more features meet significance level {significance_level}")
            break
    
    return {
        'selected_features': selected_features,
        'step_results': step_results,
        'final_model': fit_ols_model(current_X, y) if len(selected_features) > 0 else None
    }

In [ ]:
# Prepare data for stepwise regression
predictor_columns = [col for col in data.columns if col != 'Throughput']
X = data[predictor_columns].values
y = data['Throughput'].values

print(f"Predictor variables: {predictor_columns}")
print(f"Number of potential predictors: {len(predictor_columns)}")
print(f"Sample size: {len(y)}")

# Run forward stepwise regression
stepwise_results = forward_stepwise_regression(X, y, significance_level=0.05)

In [ ]:
# Display final model summary
print("\n" + "=" * 60)
print("FINAL MODEL SUMMARY")
print("=" * 60)

if stepwise_results['final_model']:
    final_model = stepwise_results['final_model']
    selected_features = stepwise_results['selected_features']
    
    print(f"Selected Features: {[data.columns[i] for i in selected_features]}")
    print(f"Intercept: {final_model['intercept']:.4f}")
    
    for i, feature_idx in enumerate(selected_features):
        feature_name = data.columns[feature_idx]
        coefficient = final_model['coefficients'][i+1]  # +1 because intercept is first
        print(f"{feature_name}: {coefficient:.4f}")
    
    print(f"\nModel Performance:")
    print(f"MSE: {final_model['mse']:.2f}")
    print(f"R-squared: {final_model['r2']:.4f}")
    print(f"AIC: {final_model['aic']:.2f}")
    print(f"BIC: {final_model['bic']:.2f}")
else:
    print("No features were selected by the stepwise procedure.")

In [ ]:
# Create comprehensive visualization of stepwise regression results
if stepwise_results['step_results']:
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Forward Stepwise Regression Analysis', fontsize=16, fontweight='bold')
    
    # Extract data for plotting
    steps = [result['step'] for result in stepwise_results['step_results']]
    r2_values = [result['r2'] for result in stepwise_results['step_results']]
    mse_values = [result['mse'] for result in stepwise_results['step_results']]
    aic_values = [result['aic'] for result in stepwise_results['step_results']]
    bic_values = [result['bic'] for result in stepwise_results['step_results']]
    f_statistics = [result['f_statistic'] for result in stepwise_results['step_results']]
    p_values = [result['p_value'] for result in stepwise_results['step_results']]
    feature_names = [result['feature_name'] for result in stepwise_results['step_results']]
    
    # Plot 1: R-squared progression
    axes[0, 0].plot(steps, r2_values, 'o-', linewidth=2, markersize=8, color='blue')
    axes[0, 0].set_xlabel('Step', fontsize=12)
    axes[0, 0].set_ylabel('R-squared', fontsize=12)
    axes[0, 0].set_title('Model Fit Improvement (R-squared)', fontsize=14, fontweight='bold')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].set_ylim(0, 1)
    
    # Add feature labels to R-squared plot
    for i, (step, r2, name) in enumerate(zip(steps, r2_values, feature_names)):
        axes[0, 0].annotate(name.split('_')[0], (step, r2), 
                           textcoords="offset points", xytext=(0,10), ha='center', fontsize=8)
    
    # Plot 2: MSE progression
    axes[0, 1].plot(steps, mse_values, 'o-', linewidth=2, markersize=8, color='red')
    axes[0, 1].set_xlabel('Step', fontsize=12)
    axes[0, 1].set_ylabel('Mean Squared Error', fontsize=12)
    axes[0, 1].set_title('Error Reduction (MSE)', fontsize=14, fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Plot 3: Information Criteria
    axes[1, 0].plot(steps, aic_values, 'o-', linewidth=2, markersize=8, label='AIC', color='green')
    axes[1, 0].plot(steps, bic_values, 's-', linewidth=2, markersize=8, label='BIC', color='orange')
    axes[1, 0].set_xlabel('Step', fontsize=12)
    axes[1, 0].set_ylabel('Information Criterion Value', fontsize=12)
    axes[1, 0].set_title('Model Selection Criteria', fontsize=14, fontweight='bold')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Plot 4: F-statistics and p-values
    ax2 = axes[1, 1]
    bars = ax2.bar(range(len(f_statistics)), f_statistics, alpha=0.7, color='purple', edgecolor='black')
    ax2.set_xlabel('Step', fontsize=12)
    ax2.set_ylabel('F-statistic', fontsize=12)
    ax2.set_title('Variable Selection Statistics', fontsize=14, fontweight='bold')
    ax2.set_xticks(range(len(f_statistics)))
    ax2.set_xticklabels([f"Step {i+1}" for i in range(len(f_statistics))])
    ax2.grid(True, alpha=0.3)
    
    # Add p-values as text on bars
    for i, (bar, f_stat, p_val) in enumerate(zip(bars, f_statistics, p_values)):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
                f'p={p_val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    # Add significance line
    ax2.axhline(y=4, color='red', linestyle='--', alpha=0.7, label='Typical F critical (α=0.05)')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("No stepwise results to visualize.")

In [ ]:
# Compare with full model and exhaustive search (small scale demonstration)
print("\n" + "=" * 60)
print("PERFORMANCE COMPARISON")
print("=" * 60)

# Full model with all predictors
X_full = np.column_stack([np.ones(len(y)), X])
full_model_results = fit_ols_model(X_full, y)

print(f"Full Model (all {len(predictor_columns)} predictors):")
print(f"  R-squared: {full_model_results['r2']:.4f}")
print(f"  MSE: {full_model_results['mse']:.2f}")
print(f"  AIC: {full_model_results['aic']:.2f}")
print(f"  BIC: {full_model_results['bic']:.2f}")

if stepwise_results['final_model']:
    stepwise_model = stepwise_results['final_model']
    print(f"\nStepwise Model ({len(stepwise_results['selected_features'])} predictors):")
    print(f"  R-squared: {stepwise_model['r2']:.4f}")
    print(f"  MSE: {stepwise_model['mse']:.2f}")
    print(f"  AIC: {stepwise_model['aic']:.2f}")
    print(f"  BIC: {stepwise_model['bic']:.2f}")
    
    # Calculate performance metrics
    r2_loss = full_model_results['r2'] - stepwise_model['r2']
    mse_increase = stepwise_model['mse'] - full_model_results['mse']
    aic_improvement = full_model_results['aic'] - stepwise_model['aic']
    bic_improvement = full_model_results['bic'] - stepwise_model['bic']
    
    print(f"\nPerformance Trade-offs:")
    print(f"  R-squared loss: {r2_loss:.4f} ({r2_loss/full_model_results['r2']*100:.2f}% loss)")
    print(f"  MSE increase: {mse_increase:.2f}")
    print(f"  AIC improvement: {aic_improvement:.2f} (lower is better)")
    print(f"  BIC improvement: {bic_improvement:.2f} (lower is better)")
    
    # Feature selection efficiency
    features_removed = len(predictor_columns) - len(stepwise_results['selected_features'])
    reduction_percentage = features_removed / len(predictor_columns) * 100
    
    print(f"\nFeature Selection Efficiency:")
    print(f"  Features removed: {features_removed}/{len(predictor_columns)} ({reduction_percentage:.1f}%)")
    print(f"  Model simplicity: Significantly improved")
    print(f"  Interpretability: Much higher with fewer variables")

# Demonstrate computational efficiency (theoretical)
print(f"\nComputational Complexity Analysis:")
p = len(predictor_columns)
exhaustive_combinations = 2**p - 1  # All non-empty subsets
stepwise_evaluations = p * (p + 1) / 2  # Approximate number of F-tests

print(f"  Exhaustive search would require: {exhaustive_combinations:,} model evaluations")
print(f"  Stepwise regression requires: ~{stepwise_evaluations:.0f} F-statistic calculations")
print(f"  Efficiency gain: {exhaustive_combinations/stepwise_evaluations:.1f}x fewer evaluations")
print(f"  Time savings: Massive for large p (exponential vs polynomial)")

In [ ]:
# Analyze variable selection correctness
print("\n" + "=" * 60)
print("VARIABLE SELECTION ANALYSIS")
print("=" * 60)

# True causal variables (from data generation)
true_causal = ['Trade_Volume', 'GDP_Growth', 'Fuel_Price', 'Seasonal_Peak', 'Ship_Density']
true_noise = ['Random_Noise_1', 'Random_Noise_2', 'Competition']

if stepwise_results['selected_features']:
    selected_names = [data.columns[i] for i in stepwise_results['selected_features']]
    
    print("True Causal Variables:")
    for var in true_causal:
        status = "✓ Selected" if var in selected_names else "✗ Missed"
        print(f"  {var}: {status}")
    
    print("\nTrue Noise Variables:")
    for var in true_noise:
        status = "✗ Correctly Excluded" if var not in selected_names else "⚠ Incorrectly Selected"
        print(f"  {var}: {status}")
    
    # Calculate selection accuracy
    true_positives = sum(1 for var in true_causal if var in selected_names)
    false_positives = sum(1 for var in true_noise if var in selected_names)
    false_negatives = sum(1 for var in true_causal if var not in selected_names)
    true_negatives = sum(1 for var in true_noise if var not in selected_names)
    
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    accuracy = (true_positives + true_negatives) / (true_positives + false_positives + false_negatives + true_negatives)
    
    print(f"\nSelection Performance Metrics:")
    print(f"  Precision: {precision:.3f} ({precision*100:.1f}% of selected variables are truly causal)")
    print(f"  Recall: {recall:.3f} ({recall*100:.1f}% of causal variables were selected)")
    print(f"  Accuracy: {accuracy:.3f} ({accuracy*100:.1f}% overall correct decisions)")
    
    if precision >= 0.8 and recall >= 0.8:
        print(f"\n✓ Excellent variable selection performance!")
    elif precision >= 0.6 and recall >= 0.6:
        print(f"\n✓ Good variable selection performance.")
else:
    print("No variables were selected for analysis.")

### Why this Tier exists vs Tier 1
Tier 2 addresses the **feature selection challenge** that Tier 1 assumes is already solved:
- **Automated variable selection** from dozens of potential predictors
- **Computational efficiency** vs exhaustive search (exponential vs polynomial time)
- **Model parsimony** - simpler, more interpretable models
- **Practical applicability** for real-world datasets with many variables

### Pros / Cons vs Tier 1
**Pros vs Tier 1:**
- **Automated feature selection** - no manual variable selection required
- **Computationally efficient** - polynomial vs exponential complexity
- **Model parsimony** - selects minimal set of important variables
- **Practical scalability** - handles dozens of potential predictors
- **Statistical rigor** - uses F-tests for significance-based selection

**Cons vs Tier 1:**
- **Local optimum problem** - greedy algorithm may miss optimal combinations
- **Order dependency** - final model can depend on variable entry order
- **No global optimization** - cannot consider all feature interactions
- **Threshold sensitivity** - results depend on significance level choice
- **Limited exploration** - cannot backtrack from poor early decisions

### When to use this Tier
- **High-dimensional data** with many potential predictors (p > 10)
- **Exploratory data analysis** to identify important variables
- **Baseline automated method** before trying more complex approaches
- **Interpretability-critical** applications requiring simple models
- **Computational constraints** where exhaustive search is infeasible
- **Real-time applications** needing fast model building